# Feature engineering — saved experiment review

This notebook reads the same curated evidence as the app. It does not fit a model, select inputs or replay a strategy. The official test was previously inspected; these are retrospective comparisons. See `../docs/FEATURE_ENGINEERING.md`.

In [1]:
import os
os.environ['MLFACTOR_READ_ONLY'] = '1'
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'published').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from published_results import PublishedStrategies
from engineering_ui import METHOD_NOTES, _curves
publication = PublishedStrategies(ROOT / 'published/project')
reader = publication.analysis_section('engineering')
assert reader is not None, 'Publish the completed engineering study first.'
training, final = reader.study('training'), reader.study('test')
training['scope'], final['scope']

({'window': 'training',
  'fit_origin': '2006-12-31',
  'trained_through_return_month': '2006-12-31',
  'fit_rows': 97788,
  'first_return_month': '2007-01-31',
  'last_return_month': '2009-12-31',
  'months': 36,
  'test_csv_loaded': False,
  'technical_missing_rows': 14589,
  'test_source_sha256': None},
 {'window': 'test',
  'fit_origin': '2009-12-31',
  'trained_through_return_month': '2009-12-31',
  'fit_rows': 140739,
  'first_return_month': '2010-01-31',
  'last_return_month': '2018-12-31',
  'months': 108,
  'test_csv_loaded': True,
  'technical_missing_rows': 14673,
  'test_source_sha256': '7291a8c3008b4c7e04509ad28b0a0bebf20433c1c75a845b3672ffa60e5a8a94'})

## Methods and input budget

Cluster averages, within-cluster PCA, IC blends, technical signals and economic composites are separate input recipes. There are at most ten classifier inputs, but some recipes aggregate twenty source factors. That distinction must be checked against the assignment rubric.

In [2]:
import pandas as pd
pd.DataFrame([{'Recipe':key,'Explanation':value} for key,value in METHOD_NOTES.items()])

,Recipe,Explanation
0,anova_10,Reference: select ten original factors using t...
1,technical_10,Ten percentile-ranked signals from already-ear...
2,cluster_average,Cluster the twenty factor columns by average l...
3,cluster_pca,"Use the same factor groups, standardize member..."
4,ic_blends,Learn each factor’s signed mean monthly Spearm...
5,economic,"Five fixed heuristics: leverage, cash flow, yi..."


## Training validation and tuning

All learned construction is fitted inside chronological folds before 2007. Both fixed and tuned models are frozen on January 2007–December 2009. The score is binary next-up/down log loss, not portfolio Sharpe. Feature IC weights learn continuous-return ranks inside each fold.

In [3]:
training['trials'][['recipe','model_key','target','params','mean_cv_log_loss','rank']].head(20)

,recipe,model_key,target,params,mean_cv_log_loss,rank
0,anova_10,hist_gradient_boosting,loss,"{""model_params"": {""learning_rate"": 0.05}}",0.678567,1
1,anova_10,hist_gradient_boosting,loss,"{""model_params"": {""learning_rate"": 0.1}}",0.679539,2
2,anova_10,hist_gradient_boosting,upside,"{""model_params"": {""learning_rate"": 0.05}}",0.681102,1
3,anova_10,hist_gradient_boosting,upside,"{""model_params"": {""learning_rate"": 0.1}}",0.682286,2
4,anova_10,logistic,loss,"{""model_params"": {""C"": 1.0}}",0.678791,2
5,anova_10,logistic,loss,"{""model_params"": {""C"": 0.1}}",0.678789,1
6,anova_10,logistic,upside,"{""model_params"": {""C"": 1.0}}",0.681205,2
7,anova_10,logistic,upside,"{""model_params"": {""C"": 0.1}}",0.681202,1
8,anova_10,naive_bayes,loss,"{""model_params"": {""var_smoothing"": 1e-09}}",0.692144,2
9,anova_10,naive_bayes,loss,"{""model_params"": {""var_smoothing"": 1e-05}}",0.692144,1


In [4]:
training['metrics'].query("book == 'K-means top 10' and target == 'loss' and model_key == 'logistic'")[[
 'recipe','status','cagr','annualized_log_growth','sharpe','max_drawdown','avg_cash_weight']]

,recipe,status,cagr,annualized_log_growth,sharpe,max_drawdown,avg_cash_weight
32,anova_10,Fixed,0.327484,0.283286,1.062665,-0.162378,0.630115
38,anova_10,Tuned,0.327485,0.283286,1.062649,-0.162383,0.630115
128,cluster_average,Fixed,0.319764,0.277453,1.062474,-0.159243,0.637546
134,cluster_average,Tuned,0.319766,0.277454,1.062473,-0.159244,0.637543
224,cluster_pca,Fixed,0.319552,0.277292,1.062000,-0.159405,0.637422
230,cluster_pca,Tuned,0.319554,0.277294,1.062000,-0.159407,0.637418
320,economic,Fixed,0.319479,0.277237,1.061990,-0.159107,0.637885
326,economic,Tuned,0.319450,0.277215,1.062006,-0.159099,0.637887
416,ic_blends,Fixed,0.322384,0.279436,1.061315,-0.160612,0.634005
422,ic_blends,Tuned,0.322389,0.279440,1.061316,-0.160614,0.634002


## Frozen final evaluation

Parameter choices come from early-training CV. Feature construction and the classifier are refitted once using training data through December 2009; no model or IC/PCA fit uses official-test labels. Lagged technical signals and stock group membership update monthly from returns already earned. Cash earns zero and results exclude trading costs.

In [5]:
final['metrics'].query("book == 'K-means top 10' and target == 'loss' and model_key == 'logistic'")[[
 'recipe','status','cagr','annualized_log_growth','sharpe','max_drawdown','avg_cash_weight']]

,recipe,status,cagr,annualized_log_growth,sharpe,max_drawdown,avg_cash_weight
32,anova_10,Fixed,0.156392,0.145305,0.589631,-0.166533,0.627542
38,anova_10,Tuned,0.156365,0.145281,0.589648,-0.166575,0.627490
128,cluster_average,Fixed,0.154213,0.143419,0.589472,-0.164781,0.631740
134,cluster_average,Tuned,0.154214,0.143419,0.589473,-0.164782,0.631738
224,cluster_pca,Fixed,0.154332,0.143522,0.589356,-0.164866,0.631489
230,cluster_pca,Tuned,0.154333,0.143523,0.589357,-0.164867,0.631487
320,economic,Fixed,0.154681,0.143824,0.587588,-0.164402,0.631729
326,economic,Tuned,0.154681,0.143824,0.587581,-0.164401,0.631728
416,ic_blends,Fixed,0.154341,0.143529,0.591504,-0.164968,0.629475
422,ic_blends,Tuned,0.154343,0.143531,0.591502,-0.164970,0.629471


In [6]:
rows = final['monthly']
rows = rows.loc[rows.book.eq('K-means top 10') & (rows.recipe.eq('original') | (
 rows.model_key.eq('logistic') & rows.target.eq('loss') & rows.status.eq('Tuned')))]
from IPython.display import HTML, display
from research.engineering_reporting import standalone_figure_html
display(HTML(standalone_figure_html(_curves(rows, 'Cumulative log return'))))

/home/sweethoney/anaconda3/lib/python3.12/site-packages/IPython/core/display.py:431: UserWarning:

Consider using IPython.display.IFrame instead



## Reproduction

The original exploratory notebook is preserved with review corrections. Canonical full-precision fitted models, features, trial/fold tables and keyed forecast panels are local research artifacts. The publication contains summaries/audits only and is sufficient to execute this notebook and the app. Formal submission release pinning remains a separate step.